# Проект №15. Аудит ошибок и объяснений классификатора изображений

**Дисциплина:** «Аналитика больших массивов» (В22.045) · Yessenov University · 4 курс, 2026–2027

Этот ноутбук показывает весь проект на открытом датасете **Oxford-IIIT Pet** (7 349 фото, 37 пород кошек и собак, у каждого фото есть маска животного):

1. Загрузка и просмотр датасета
2. Паспорт данных и сплит без утечек
3. Модели и качество на тесте (обученная модель берётся из репозитория)
4. Grad-CAM — куда смотрит сеть
5. Подмена фона в реальном времени
6. Остальные результаты: проверка карт, водяной знак, атлас ошибок, скорость
7. Проверка на своём фото

**Как запустить:** `Среда выполнения → Выполнить все` (лучше включить GPU: `Среда выполнения → Сменить среду → T4 GPU`). Первый запуск занимает 5–10 минут: скачивается датасет (~800 МБ).

Код: https://github.com/Ampldariik/xai-audit-oxford-pets

## 0. Подготовка среды

In [ ]:
import os, sys, json
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Ampldariik/xai-audit-oxford-pets'
ROOT = '/content/xai-audit-oxford-pets' if IN_COLAB else os.environ.get('XAI_ROOT', os.getcwd())

if IN_COLAB:
    if not os.path.exists(ROOT):
        !git clone -q {REPO_URL} {ROOT}
    !pip -q install grad-cam captum imagehash

os.chdir(ROOT)
sys.path.insert(0, ROOT)

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, Markdown, display
plt.rcParams['figure.dpi'] = 110
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| устройство:', DEVICE)

## 1. Датасет Oxford-IIIT Pet

Скачиваем открытый датасет через `torchvision` (источник — VGG, Оксфорд; лицензия CC BY-SA 4.0).
В нём фотографии (`images/`) и разметка (`annotations/`): список классов и **trimap-маски** — где на фото животное, а где фон.

In [ ]:
from torchvision.datasets import OxfordIIITPet

if not os.path.exists('data/raw/images'):
    OxfordIIITPet(root='data/download', split='trainval', target_types='segmentation', download=True)
    os.makedirs('data/raw', exist_ok=True)
    for d in ('images', 'annotations'):
        os.symlink(os.path.abspath(f'data/download/oxford-iiit-pet/{d}'), f'data/raw/{d}')

from src.data import read_annotations, class_names
ann = read_annotations('data/raw')
NAMES = class_names(ann)
print(f'Фотографий: {len(ann)}, пород: {ann.label.nunique()}')
print(ann.species.value_counts().rename({'dog': 'собак', 'cat': 'кошек'}).to_dict())
ann.head()

In [ ]:
# сколько фото на каждую породу
cnt = ann.groupby(['class_name', 'species']).size().reset_index(name='n').sort_values('n')
plt.figure(figsize=(12, 3.5))
plt.bar(cnt.class_name, cnt.n, color=['#D9792B' if s == 'cat' else '#2F6FDE' for s in cnt.species])
plt.xticks(rotation=90, fontsize=7); plt.ylabel('фото'); plt.title('Фото на породу (оранжевый — кошки, синий — собаки)')
plt.tight_layout(); plt.show()

In [ ]:
# примеры: фото и маска (жёлтый — граница, фиолетовый — животное, бирюзовый — фон)
from PIL import Image
rng = np.random.default_rng(0)
sample = ann.sample(6, random_state=1)
fig, ax = plt.subplots(2, 6, figsize=(16, 5.5))
for k, r in enumerate(sample.itertuples()):
    ax[0, k].imshow(Image.open(f'data/raw/images/{r.image_id}.jpg').convert('RGB')); ax[0, k].set_title(r.class_name, fontsize=9)
    ax[1, k].imshow(Image.open(f'data/raw/annotations/trimaps/{r.image_id}.png'))
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

## 2. Паспорт данных и сплит без утечек

Скрипт `01_prepare_data.py`:
- приводит все фото и маски к 224×224 и складывает в быстрый кэш;
- считает **pHash** («отпечаток» картинки) и объединяет почти одинаковые фото в одну серию;
- делит данные 70/15/15 так, чтобы серия целиком попадала в одну часть (`StratifiedGroupKFold`) — **без утечки** между обучением и тестом.

In [ ]:
import shutil
shutil.copy('data/splits.csv', 'data/splits_repo.csv')   # сплит, на котором обучались модели из репозитория
!python scripts/01_prepare_data.py
new, old = pd.read_csv('data/splits.csv'), pd.read_csv('data/splits_repo.csv')
if (new.split.values == old.split.values).all():
    print('Сплит воспроизвёлся один в один с тем, на котором обучались модели.')
else:
    shutil.copy('data/splits_repo.csv', 'data/splits.csv')
    print('Сплит отличается из-за версий библиотек — вернул исходный, чтобы тест не пересекался с обучением.')

In [ ]:
passport = json.load(open('reports/data_passport.json', encoding='utf-8'))
for k in ['task', 'target_variable', 'unit_of_observation', 'source', 'license', 'n_images', 'n_classes', 'split_counts', 'near_duplicates']:
    print(f'{k:22s}: {passport[k]}')

In [ ]:
display(ShowImage('reports/figures/eda_near_duplicates.png', width=1000))

## 3. Модели и качество на тесте

Сравниваем 4 модели. Обученные веса лежат в репозитории (`runs/`), поэтому заново обучать не нужно — скрипт `03_evaluate.py` прямо сейчас пересчитывает метрики на отложенном тесте (1 050 фото).

| Модель | Что это |
|---|---|
| Baseline-0 | логистическая регрессия, видит только цвета |
| Baseline-1 | маленькая свёрточная сеть, обученная с нуля |
| Linear probe | готовые признаки ResNet-18 + логистическая регрессия |
| **Основная** | ResNet-18 (ImageNet), дообученная на наших породах |

In [ ]:
!python scripts/03_evaluate.py 2>&1 | grep -v Warning

In [ ]:
cmp = pd.read_csv('reports/model_comparison.csv')
cmp[['model', 'macro_f1', 'macro_f1_ci95', 'accuracy', 'top5_accuracy', 'species_accuracy']].round(3)

In [ ]:
display(ShowImage('reports/figures/confusion_main.png', width=750))
display(pd.read_csv('reports/top_confusions_main.csv').head(7))

## 4. Grad-CAM: куда смотрит сеть

Grad-CAM строит тепловую карту: красным — области, которые сильнее всего повлияли на ответ.
**FG** — доля карты, которая попала на животное (по маске). Чем ближе к 1, тем больше сеть смотрит на само животное, а не на фон.

In [ ]:
from src.models import load_checkpoint, gradcam_target_layer
from src.data import to_tensor_batch
from src.xai import gradcam_maps, attention_metrics, overlay

df = pd.read_csv('data/splits.csv')
imgs = np.load('data/cache/images_224.npy', mmap_mode='r')
msks = np.load('data/cache/masks_224.npy', mmap_mode='r')
model, _ = load_checkpoint('runs/main/model.pt', DEVICE)
test_idx = np.where(df.split == 'test')[0]

def predict(arr):
    with torch.no_grad():
        return torch.softmax(model(to_tensor_batch(arr).to(DEVICE)), 1).cpu().numpy()

sel = np.random.default_rng(7).choice(test_idx, 8, replace=False)
arr = np.array(imgs[sel]); p = predict(arr); pred = p.argmax(1)
cams = gradcam_maps(model, gradcam_target_layer(model), to_tensor_batch(arr).to(DEVICE), pred)
fig, ax = plt.subplots(2, 8, figsize=(20, 5.5))
for k, i in enumerate(sel):
    m = attention_metrics(cams[k], msks[i])
    ok = pred[k] == df.label[i]
    ax[0, k].imshow(arr[k]); ax[0, k].set_title(f'{df.class_name[i]}', fontsize=8)
    ax[1, k].imshow(overlay(arr[k], cams[k])); ax[1, k].contour((msks[i] != 2).astype(float), [0.5], colors='white', linewidths=0.6)
    ax[1, k].set_title(f"{'ВЕРНО' if ok else 'ОШИБКА'}: {NAMES[pred[k]]}\n p={p[k].max():.2f}  FG={m['fg_energy']:.2f}", fontsize=8, color='green' if ok else 'red')
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
xs = json.load(open('reports/xai_summary.json'))
print(f"FG-energy основной модели: {xs['main']['fg_energy_mean']:.2f}  (сеть с нуля: {xs['baseline_cnn']['fg_energy_mean']:.2f})")
print(f"Pointing game: {xs['main']['pointing_game']:.1%}")
sc = json.load(open('reports/saliency_check.json'))
d = sc['deletion']
print(f"Проверка карт: закрыли топ-20% Grad-CAM → падение p = {d['top']['mean_drop_p_pred']:.3f}; случайные 20% → {d['random']['mean_drop_p_pred']:.3f}")
print(f"Рандомизация весов: медиана корреляции карт = {sc['model_randomization']['spearman_median']:.2f} (тест пройден: {sc['model_randomization']['passed']})")
display(ShowImage('reports/figures/saliency_deletion.png', width=1000))

## 5. Подмена фона в реальном времени

Животное остаётся тем же **пиксель в пиксель**, меняется только фон. Если ответ модели меняется — значит, она смотрела на фон.
Фоны берутся с фото из валидации: животное вырезается по маске и закрашивается (inpainting).

In [ ]:
from src.background import remove_object, composite
import cv2

val_idx = np.where(df.split == 'val')[0]
plates = [remove_object(np.ascontiguousarray(imgs[j]), msks[j]) for j in np.random.default_rng(1).choice(val_idx, 3, replace=False)]

show = np.random.default_rng(3).choice(test_idx, 4, replace=False)
fig, ax = plt.subplots(4, 6, figsize=(17, 12))
for r, i in enumerate(show):
    im, tm = np.ascontiguousarray(imgs[i]), msks[i]
    variants = {'исходное': im,
                'серый фон': composite(im, tm, np.full_like(im, 128)),
                'размытый фон': composite(im, tm, cv2.GaussianBlur(im, (0, 0), 12)),
                'чужой фон 1': composite(im, tm, plates[0]),
                'чужой фон 2': composite(im, tm, plates[1]),
                'без животного': remove_object(im, tm)}
    pv = predict(np.stack(list(variants.values())))
    for c, (name, v) in enumerate(variants.items()):
        k = pv[c].argmax(); ok = k == df.label[i]
        ax[r, c].imshow(v); ax[r, c].axis('off')
        ax[r, c].set_title(f'{name}\n{NAMES[k]} ({pv[c].max():.2f})', fontsize=8, color='green' if ok else 'red')
plt.tight_layout(); plt.show()

In [ ]:
bg = pd.read_csv('reports/bg_swap_metrics.csv')
bg['flip_rate_%'] = (bg.flip_rate * 100).round(1)
bg.pivot(index='variant', columns='model', values='flip_rate_%').loc[
    ['fg_gray', 'bg_blur', 'bg_swap_random', 'bg_swap_species', 'bg_only']].rename_axis('доля фото, у которых сменился ответ, %')

## 6. Остальные результаты проекта

### 6.1 Водяной знак (положительный контроль)
На все обучающие фото питбулей поставили надпись «(c) PETSTOCK» и обучили новую модель. Проверка нашла ярлык: Grad-CAM показывает второй очаг внимания на надписи.

In [ ]:
w = json.load(open('reports/watermark_results_after_aug.json'))['poisoned']
print(f"Recall питбуля: без надписи {w['target_recall_clean']:.2f} → с надписью {w['target_recall_wm']:.2f}")
print(f"Доля Grad-CAM в зоне надписи: {w['gradcam_box_energy_target_imgs']:.1%} (площадь зоны {w['box_area_frac']:.0%})")
display(ShowImage('reports/figures/watermark_target_after_aug.png', width=1000))

### 6.2 Атлас ошибок

In [ ]:
display(Markdown(open('reports/error_atlas.md', encoding='utf-8').read()))
display(ShowImage('reports/figures/atlas_counts.png', width=900))
display(ShowImage('reports/figures/atlas_fine_grained.png', width=1000))

### 6.3 Масштаб: время и память на N = 100 и N = 1000

In [ ]:
agg = pd.read_csv('reports/benchmark_agg.csv')
agg[['N', 'batch_size', 'total_s_mean', 'throughput_img_s_mean', 'gradcam_s_mean', 'rss_increase_mb_mean']].round(2)

In [ ]:
display(ShowImage('reports/figures/benchmark.png', width=1000))

## 7. Проверка на своём фото

Загрузите фото кошки или собаки — модель назовёт породу (топ-3) и покажет Grad-CAM. Работает только в Colab.

In [ ]:
if IN_COLAB:
    from google.colab import files
    up = files.upload()
    for fname in up:
        im = np.asarray(Image.open(fname).convert('RGB').resize((224, 224)))
        p = predict(im[None])[0]; top = p.argsort()[::-1][:3]
        cam = gradcam_maps(model, gradcam_target_layer(model), to_tensor_batch(im[None]).to(DEVICE), [top[0]])[0]
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(im); ax[1].imshow(overlay(im, cam))
        for a in ax: a.axis('off')
        ax[0].set_title('\n'.join(f'{NAMES[t]}: {p[t]:.2f}' for t in top), fontsize=9)
        ax[1].set_title('Grad-CAM'); plt.tight_layout(); plt.show()
else:
    print('Загрузка своего фото работает в Google Colab')

## 8. (Необязательно) Полный перезапуск проекта

Все этапы с нуля, включая обучение всех моделей. На GPU T4 — около 30–40 минут. Снимите `#`, чтобы запустить.

In [ ]:
# !bash run_all.sh